# 01_01 Tokens: where does one word end and the next begin?

Every NLP program starts by cutting text into **tokens**. It sounds like the easy part. By the end of this
notebook you will have watched two sentence splitters agree on the number of sentences in a notice and both
put the boundaries in the wrong place, four word tokenizers disagree about one short sentence, and the real
tokenizers of two real language models read *Kittiwake* as four pieces and a price as single digits.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-01-why-cant-a-computer-read", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'spacy': 'spacy',
           'nltk': 'nltk',
           'transformers': 'transformers',
           'en_core_web_sm': 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import re
import json
import nltk
import spacy
from transformers import AutoTokenizer
from nlpcheck import ask, guess, reveal, check_01_01

nlp = spacy.load("en_core_web_sm")   # spaCy's small English pipeline, already in the image
# The tokenizers of two real models, from Hugging Face, stored in the image (nothing downloads):
bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")        # BERT, Lab 11
smol = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")  # a small language model, Lab 12
notices = open("data/kittiwake_notices.txt").read().strip().split("\n\n")
print(len(notices), "Kittiwake service notices loaded")

## 1. Recall

From the chapter pages. Answer from memory, then run the cell.

**r1.** "Visiting relatives can be boring" is an example of which kind of ambiguity?
(a) lexical, (b) syntactic, (c) referential

**r2.** Which step has to come first? (a) stemming, (b) tagging, (c) tokenization

In [ ]:
ask("r1", "")   # put your letter between the quotes
ask("r2", "")

## 2. Sentences: is a full stop the end of a sentence?

Here is the first notice, the one about the planned maintenance.

In [ ]:
maintenance = notices[0]
print(maintenance)

Read it as a person: how many sentences are there? Then predict how many pieces you get by the obvious
program, splitting wherever a full stop is followed by a space.

In [ ]:
guess("naive_sentences", None)   # replace None with a number

In [ ]:
naive = [s for s in maintenance.split(". ") if s]
for s in naive:
    print("|", s)
reveal("naive_sentences", len(naive))

Seven. Every abbreviation (`St.`, `a.m.`, `approx.`) looks like the end of a sentence to that rule.
A person reads five: the long first one, "Calls and data may drop", "Customers ... meantime", "Questions?"
and "Text HELP ...".

NLTK's `sent_tokenize` uses **Punkt**, a model that learned from a large body of English which words are
usually abbreviations. Predict: how many sentences will it find?

In [ ]:
guess("punkt_sentences", None)

In [ ]:
punkt = nltk.sent_tokenize(maintenance)
for s in punkt:
    print("|", s)
reveal("punkt_sentences", len(punkt))

Five, the right number, and **two of the boundaries are wrong**. Punkt has never seen `approx.`, so it
ended a sentence there. It does know `a.m.`, so when "5 a.m." really did end a sentence it carried straight
on into "Calls". The two mistakes cancel out in the count, which is exactly why you look at the sentences
and not just at how many there are.

spaCy finds sentence boundaries a different way, from its parse of the whole text. See whether it does
better:

In [ ]:
for s in nlp(maintenance).sents:
    print("|", s.text)

The same two mistakes. An abbreviation at the end of a sentence is genuinely ambiguous: the full stop is
doing two jobs at once. You can teach Punkt one more abbreviation, which fixes one mistake and leaves the
other:

In [ ]:
from nltk.tokenize.punkt import PunktTokenizer
punkt_plus = PunktTokenizer()
punkt_plus._params.abbrev_types.add("approx")   # Punkt stores abbreviations lowercase, without the dot
for s in punkt_plus.tokenize(maintenance):
    print("|", s)

Four sentences now: "approx. 5 a.m." stays together, and "Calls" is still glued on. Production systems
that care (legal documents, clinical notes) add rules for their own domain on top of a model like this.

## 3. Words: how many tokens are in "I can't pay $42.50 in the U.S. until Friday."?

In [ ]:
s = "I can't pay $42.50 in the U.S. until Friday."
guess("split_tokens", None)   # how many pieces from s.split()?
guess("nltk_tokens", None)    # and from nltk.word_tokenize(s)?

In [ ]:
print("split :", s.split())
print("NLTK  :", nltk.word_tokenize(s))
print("regex :", re.findall(r"\w+", s))
reveal("split_tokens", len(s.split()))
reveal("nltk_tokens", len(nltk.word_tokenize(s)))

- `split()` cuts on spaces only, so `Friday.` keeps its full stop and `can't` stays whole. Nine tokens.
- NLTK follows the **Penn Treebank** convention: `can't` becomes `ca` and `n't`, so the negation is a token
  of its own that a later step can find; the dollar sign is separated from the amount; `U.S.` survives as one
  token because it is a known abbreviation. Twelve.
- The regular expression `\w+` keeps only runs of letters and digits, and tears `U.S.`, `42.50` and
  `can't` apart. It is the tokenizer in the book's Chapter 2 notebook, so remember what it does to money.

## 4. spaCy: tokens are objects, not strings

In [ ]:
doc = nlp(s)
for t in doc:
    print(f"{t.text:8} punct={t.is_punct!s:5} number={t.like_num!s:5} currency={t.is_currency!s:5} space_after={t.whitespace_!r}")

spaCy splits the sentence exactly as NLTK does, but each token carries facts about itself, and remembers
the whitespace after it, so `doc.text` rebuilds the original sentence character for character. NLTK hands
you strings and the original spacing is gone.

## 5. How language models tokenize

Language models do not use a word tokenizer. Each one ships with its own **subword** tokenizer: a fixed
vocabulary of pieces, learned from a huge body of text, so that common words are one piece and rare words
are spelled out from several. Hugging Face's `AutoTokenizer` loads the exact tokenizer a model was trained
with, which matters: a model only understands the pieces it learned.

Two are loaded above. **BERT** (the model of Chapter 11) uses **WordPiece**: a piece that continues a word
starts with `##`. **SmolLM2**, a small language model of the kind Lab 12 runs, uses **byte-level byte pair
encoding (BPE)**: a piece that follows a space starts with `Ġ`, which is how the tokenizer writes a space.

Predict how many pieces BERT makes of `Kittiwake`, and how many pieces SmolLM2 makes of `$42.50`.

In [ ]:
guess("bert_kittiwake", None)
guess("smol_price", None)

In [ ]:
for w in ["the", "Kittiwake", "can't", "tokenization", "unbelievably", "Gullhaven"]:
    print(f"{w!r:16} BERT {bert.tokenize(w)!s:42} SmolLM2 {smol.tokenize(w)}")
reveal("bert_kittiwake", len(bert.tokenize("Kittiwake")))
reveal("smol_price", len(smol.tokenize(" $42.50")))
print(smol.tokenize(" $42.50"))
print("vocabulary sizes:", bert.vocab_size, "and", smol.vocab_size)

Four pieces for `Kittiwake` in both, split in different places: a carrier invented for this course is not
in either vocabulary, so both spell it out. And SmolLM2 reads `$42.50` as six pieces: the space and dollar
sign, then **one digit at a time**. Many recent models split numbers into single digits on purpose, so that
every number is made of the same ten symbols; it is part of why arithmetic is hard for them.

The trade subword tokenization makes: no word is ever unknown, because anything can be spelled out in
pieces, and the price is that rare words and names cost more pieces. Models are limited and billed in
tokens, not words.

Now the sentence from section 3, as each model sees it:

In [ ]:
print("BERT   ", len(bert.tokenize(s)), bert.tokenize(s))
print("SmolLM2", len(smol.tokenize(s)), smol.tokenize(s))
print("BERT ids:", bert(s)["input_ids"])

Two things to notice. BERT's tokenizer is **uncased**: it lowercased everything before it split, so to BERT
"Friday" and "friday" are the same. And the numbers in the last line are what the model actually receives:
each piece is replaced by its position in the vocabulary, with `101` and `102` (`[CLS]` and `[SEP]`) added at
the ends. Turning words into numbers is the whole of the next lab.

## 6. Your turn: the outage notice

Worked above on the maintenance notice; now the third notice, the outage update (`notices[2]`). Fill in the
three lines marked `# YOUR CODE HERE`:

- `sentences`: the list of sentences NLTK finds,
- `words`: the list of NLTK word tokens for the whole notice,
- `lm_tokens`: how many pieces SmolLM2's tokenizer makes of the whole notice.

Then run the save cell and the check.

In [ ]:
outage = notices[2]
print(outage)

sentences = None    # YOUR CODE HERE
words = None        # YOUR CODE HERE
lm_tokens = None    # YOUR CODE HERE

In [ ]:
import os
os.makedirs("out", exist_ok=True)
with open("out/01_01_tokens.json", "w") as f:
    json.dump({"sentences": sentences, "words": words, "lm_tokens": lm_tokens}, f, indent=1)
print("saved out/01_01_tokens.json")
check_01_01()

## 7. Exit ticket

Explain it back, in your own words, in the cell below: why did two tokenizers find the right *number* of
sentences in the maintenance notice and still get it wrong? One or two sentences. Writing it down is the
part that makes it stick.

*Your explanation:* 